# PHYS 132 Activity 2: Where the Sun makes its energy, and the virial theorem

We use a standard solar model, BS05(OP) of Bahcall, Serenelli and Basu (2005, astro-ph/0412440), which is a
numerical solution of the equations of stellar structure for the Sun as it is today, tabulated in 1268 shells
from the centre out to $0.983\,R_\odot$. For each shell the table gives the mass $m$ inside it, its radius $r$,
the temperature $T$, density $\rho$ and pressure $P$, the luminosity $L$ flowing out through it, and the mass
fractions of hydrogen ($X$), helium ($Y$) and a few heavier nuclei. Everything is in cgs units, as in the
lecture notes.

Run the first cell as it is. In each part after that, fill in the lines marked `...` and run the cell. The
questions at the end of each part are for your group to discuss and to answer in a sentence or two.

In [ ]:
# Run this cell first. It loads the solar model and defines the constants we need, in cgs units.
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid, trapezoid
np.seterr(divide="ignore", invalid="ignore")   # the innermost shells have L = 0 and repeated values

URL = "https://www.sns.ias.edu/~jnb/SNdata/Export/BS2005/bs05op.dat"
if not os.path.exists("bs05op.dat"):
    # If this download fails, get bs05op.dat from Canvas and upload it next to this notebook.
    urllib.request.urlretrieve(URL, "bs05op.dat")

rows = []
for line in open("bs05op.dat"):
    words = line.split()
    if len(words) == 12 and words[0][0].isdigit():
        rows.append([float(w) for w in words])
table = np.array(rows)

G    = 6.674e-8      # gravitational constant, cm^3 g^-1 s^-2
a    = 7.566e-15     # radiation constant, erg cm^-3 K^-4
c    = 2.998e10      # speed of light, cm s^-1
k_B  = 1.381e-16     # Boltzmann constant, erg K^-1
m_u  = 1.661e-24     # atomic mass unit, g
Msun = 1.989e33      # g
Rsun = 6.9598e10     # cm     (the solar radius and luminosity used to build the model)
Lsun = 3.8418e33     # erg s^-1

m   = table[:, 0] * Msun    # mass inside the shell, g
r   = table[:, 1] * Rsun    # radius of the shell, cm
T   = table[:, 2]           # temperature, K
rho = table[:, 3]           # density, g cm^-3
P   = table[:, 4]           # pressure, dyn cm^-2
L   = table[:, 5] * Lsun    # luminosity flowing out through the shell, erg s^-1
X   = table[:, 6]           # hydrogen mass fraction
Y   = table[:, 7]           # helium-4 mass fraction

print(f"{len(r)} shells, from r = {r[0]/Rsun:.4f} R_sun to r = {r[-1]/Rsun:.3f} R_sun")

## 1. Where the luminosity is generated

The cell plots the luminosity flowing out through each shell, as a fraction of $L_\odot$, against the mass
inside the shell and against radius. Find the mass and the radius inside which the Sun generates 50%, 90% and
99% of its luminosity. `np.interp(x, xp, fp)` returns the value of `fp` at the point where `xp` equals `x`,
interpolating linearly between table entries.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(m / Msun, L / Lsun); axes[0].set_xlabel(r"$m/M_\odot$")
axes[1].plot(r / Rsun, L / Lsun); axes[1].set_xlabel(r"$r/R_\odot$")
for ax in axes:
    ax.set_ylabel(r"$L/L_\odot$")
plt.tight_layout()
plt.show()

for f in [0.50, 0.90, 0.99]:
    m_f = ...   # mass (in solar masses) inside which L reaches f L_sun
    r_f = ...   # the same, as a radius in solar radii
    print(f"{f:.0%} of L_sun is generated inside m = {m_f:.3f} M_sun, r = {r_f:.3f} R_sun")

What fraction of the Sun's mass, and what fraction of its volume, takes part in generating the
luminosity? What is the rest of the star doing?

## 2. The energy generation rate

Energy conservation in a shell, from the lecture, gives $dL/dm = \varepsilon - \varepsilon_\nu - T\,ds/dt$. The
$T\,ds/dt$ term is small in the present Sun, so $dL/dm$ is the rate at which nuclear reactions heat the gas,
per unit mass, after the neutrinos have taken their share.

The table gives $L/L_\odot$ to only five decimal places, and its shells are packed very closely near the
centre, so the difference between neighbouring shells is dominated by rounding. The function `rate` below
interpolates $L$ onto bins of equal mass, $0.005\,M_\odot$ wide, and differentiates across each bin; it
returns $dL/dm$ and the mass at the middle of each bin.

In [ ]:
def rate(L, m, dm=0.005 * Msun, m_max=0.6 * Msun):
    """dL/dm across bins of equal mass dm out to m_max, and the mass at the middle of each bin."""
    edges = np.arange(0, m_max + dm / 2, dm)
    L_edges = np.interp(edges, m, L)
    return np.diff(L_edges) / dm, 0.5 * (edges[1:] + edges[:-1])

eps, mmid = rate(L, m)                                    # erg g^-1 s^-1
rm, Tm, rhom, Xm = [np.interp(mmid, m, q) for q in (r, T, rho, X)]   # r, T, rho, X at the bin centres

eps_c = ...                                  # the energy generation rate in the innermost bin
eps_mean = ...                               # the Sun's average, L_sun / M_sun
print(f"central rate {eps_c:.3g} erg/g/s, average over the Sun {eps_mean:.3g} erg/g/s")

plt.plot(rm / Rsun, eps, ".")
plt.xlim(0, 0.4)
plt.xlabel(r"$r/R_\odot$")
plt.ylabel(r"$dL/dm$ (erg g$^{-1}$ s$^{-1}$)")
plt.show()

How does the central rate compare with the Sun's average? Compare both with the heat your own body
produces, roughly 100 W for 70 kg, converted to erg g$^{-1}$ s$^{-1}$.

## 3. Temperature sensitivity

In the lecture we saw that for reactions between two protons $\varepsilon \propto \rho X^2\,\mathcal{F}(T)$,
where $\mathcal{F}(T)$ rises steeply with temperature and is derived in November. Over the narrow range of
temperature in the solar core, write it as a power law,
$$
\varepsilon \approx \varepsilon_0\, \rho X^2\, T^\nu .
$$
You can estimate $\nu$ from two points before fitting anything: take $T$ and $\varepsilon/\rho X^2$ in the
innermost bin and at $0.2\,R_\odot$, and divide the logarithm of the factor by which $\varepsilon/\rho X^2$ falls
by the logarithm of the factor by which $T$ falls. Then check the estimate by fitting a straight line to
$\ln(\varepsilon/\rho X^2)$ against $\ln T$ inside $0.25\,R_\odot$, where nearly all of the luminosity is
generated. `np.polyfit(x, y, 1)` returns the slope and intercept of the best-fit line.

In [ ]:
y = ...                                      # eps / (rho X^2) in every bin
j = np.argmin(abs(rm - 0.2 * Rsun))          # the bin at 0.2 R_sun; the innermost bin is 0
nu_estimate = ...                            # from bins 0 and j, before any fitting

core = (rm < 0.25 * Rsun) & (eps > 0)
nu, ln_eps0 = np.polyfit(np.log(Tm[core]), np.log(y[core]), 1)
print(f"two-point estimate nu = {nu_estimate:.2f}, fitted nu = {nu:.2f}")

plt.plot(np.log(Tm[core]), np.log(y[core]), ".", label="model")
plt.plot(np.log(Tm[core]), nu * np.log(Tm[core]) + ln_eps0, label=f"slope {nu:.2f}")
plt.xlabel(r"$\ln T$")
plt.ylabel(r"$\ln(\varepsilon/\rho X^2)$")
plt.legend()
plt.show()

How well do the two-point estimate and the fit agree? With your fitted $\nu$, by what factor does $\varepsilon$
change if the temperature rises by 10%? Use that to explain the size of the energy-generating region you found in part 1.

# Part B

Parts 4 and 5 open the next class. If your group finished parts 1 to 3, you are welcome to start them now.

## 4. The virial theorem in the model

In Lecture 3 we combined the virial theorem of Lecture 2, $3\int P\,dV = -E_{\rm g}$ with
$E_{\rm g} = -\int_0^M Gm\,dm/r = -\alpha GM^2/R$, with the ideal gas law, and found
$\langle T\rangle = (\alpha/3)\,G\bar m M/k_BR$; at the start of this lecture the transport equation gave
$T_c \ge \langle T\rangle$. The model lets us check
every step. Since $dV = dm/\rho$, the left side of the virial theorem is $\int_0^M (3P/\rho)\,dm$; compute it,
compute $E_{\rm g}$, and find $\alpha$ for the Sun (a sphere of uniform density has $\alpha = 3/5$).
`trapezoid(y, x)` integrates `y` over `x`.

For an ideal gas $3P/\rho = 3k_BT/\bar m$, so the same integral is a mass-weighted mean temperature. The mean
mass per particle of fully ionised gas is $\bar m = m_u/(2X + \tfrac34 Y + \tfrac12 Z)$, with $Z = 1 - X - Y$;
we derive this in Week 3.

In [ ]:
E_g = ...              # -integral of G m / r dm
pressure_side = ...    # integral of 3 P / rho dm
alpha = -E_g * Rsun / (G * Msun**2)
print(f"3 int P dV / (-E_g) = {pressure_side / -E_g:.5f},  alpha = {alpha:.3f}")

Z = 1 - X - Y
mbar = ...             # mean mass per particle in each shell
T_mean = trapezoid(T, m) / m[-1]                          # mass-weighted mean temperature
print(f"mbar/m_u: {mbar[0]/m_u:.3f} at the centre, {mbar[-1]/m_u:.3f} in the outermost shell")
print(f"mass-weighted <T> = {T_mean:.3g} K, T_c = {T[0]:.3g} K, T_c/<T> = {T[0]/T_mean:.2f}")
print(f"<k T / mbar> = {trapezoid(k_B * T / mbar, m) / m[-1]:.4g},  alpha G M / 3R = {alpha * G * Msun / (3 * Rsun):.4g}")

1. How well does the model satisfy the virial theorem?
2. How does $\alpha$ for the Sun compare with $3/5$, and what does that tell you about how the mass is
   distributed?
3. By what factor is the centre hotter than the mass-weighted mean? And why is $\bar m$ larger at the centre
   than at the surface?

## 5. If you finish early: the surface boundary conditions

In the lecture we closed the structure equations with $P = 0$ and $T = 0$ at the surface. The table stops at
$0.983\,R_\odot$. Compare $P$ and $T$ in the outermost shell with their central values, and find the fraction
of the Sun's mass that lies outside it. How good an approximation are the zero boundary conditions, judged
against the interior?

In [ ]:
print(f"P_out/P_c = {...:.2e}")
print(f"T_out/T_c = {...:.2e}")
print(f"mass outside the last shell: {...:.1e} M_sun")